# 03a - Historical Data Preparation

## Objective

This notebook prepares the historical CAMS solar-radiation dataset for subsequent statistical analysis of solar availability anomalies and drought events.

The purpose of this stage is to determine whether the historical dataset is sufficiently complete, temporally consistent, and internally coherent for constructing a historical solar-availability time series.

The preparation workflow includes:

1. Loading the historical CAMS solar-radiation dataset.
2. Parsing and validating observation periods.
3. Establishing the exact temporal coverage of the downloaded dataset.
4. Checking hourly temporal continuity and duplicate observations.
5. Assessing missing values and numerical data quality.
6. Evaluating the reliability information provided by CAMS.
7. Checking physical and component-level consistency of the radiation variables.
8. Separating nighttime and solar-active observations.
9. Applying the clear-sky normalization methodology established in Notebook 03.
10. Examining the resulting historical Clear-Sky Index (CSI) series.
11. Separating the primary 2020 analysis period from observations outside that period.

## Research Principle

This notebook performs data preparation and quality assessment only.

No anomaly threshold or solar-drought threshold is defined here.

Observations are not removed solely because they exhibit unusually low solar availability. Potential quality issues are instead identified using explicit diagnostic criteria.

The historical dataset will provide the input for the anomaly analysis performed in Notebook 04.

## Dataset

The historical dataset used in this notebook was downloaded from the CAMS Solar Radiation Time-Series service for the Davos measurement location.

The requested location is:

- Latitude: 46.80° N
- Longitude: 9.83° E
- Altitude: 1610 m

The dataset contains hourly solar-radiation observations and corresponding clear-sky reference values.

The primary radiation variable used for solar-availability analysis is Global Horizontal Irradiation (GHI).

The corresponding clear-sky GHI is used as the reference for calculating the Clear-Sky Index (CSI):

$$
CSI_t =
\frac{GHI_{\text{observed},t}}
{GHI_{\text{clear},t}}
$$

This normalization methodology was established and validated using the January 2020 pilot dataset in Notebook 03.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
data_path = Path(
    "../data/raw/solar/cams_davos_2020_hourly.csv"
)

print("File exists:", data_path.exists())
print("File path:", data_path)

File exists: True
File path: ..\data\raw\solar\cams_davos_2020_hourly.csv


In [3]:
df = pd.read_csv(
    data_path,
    sep=";",
    comment="#",
    header=None,
    names=[
        "observation_period",
        "toa",
        "clear_sky_ghi",
        "clear_sky_bhi",
        "clear_sky_dhi",
        "clear_sky_bni",
        "ghi",
        "bhi",
        "dhi",
        "bni",
        "reliability",
    ],
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (9528, 11)

Columns:
['observation_period', 'toa', 'clear_sky_ghi', 'clear_sky_bhi', 'clear_sky_dhi', 'clear_sky_bni', 'ghi', 'bhi', 'dhi', 'bni', 'reliability']


In [4]:
print("========== FIRST OBSERVATIONS ==========")
print(df.head(5).to_string())

========== FIRST OBSERVATIONS ==========
                            observation_period  toa  clear_sky_ghi  clear_sky_bhi  clear_sky_dhi  clear_sky_bni  ghi  bhi  dhi  bni  reliability
0  2020-01-01T00:00:00.0/2020-01-01T01:00:00.0  0.0            0.0            0.0            0.0            0.0  0.0  0.0  0.0  0.0          1.0
1  2020-01-01T01:00:00.0/2020-01-01T02:00:00.0  0.0            0.0            0.0            0.0            0.0  0.0  0.0  0.0  0.0          1.0
2  2020-01-01T02:00:00.0/2020-01-01T03:00:00.0  0.0            0.0            0.0            0.0            0.0  0.0  0.0  0.0  0.0          1.0
3  2020-01-01T03:00:00.0/2020-01-01T04:00:00.0  0.0            0.0            0.0            0.0            0.0  0.0  0.0  0.0  0.0          1.0
4  2020-01-01T04:00:00.0/2020-01-01T05:00:00.0  0.0            0.0            0.0            0.0            0.0  0.0  0.0  0.0  0.0          1.0


In [5]:
print("========== LAST OBSERVATIONS ==========")
print(df.tail(5).to_string())

========== LAST OBSERVATIONS ==========
                               observation_period  toa  clear_sky_ghi  clear_sky_bhi  clear_sky_dhi  clear_sky_bni  ghi  bhi  dhi  bni  reliability
9523  2021-01-31T19:00:00.0/2021-01-31T20:00:00.0  0.0            0.0            0.0            0.0            0.0  0.0  0.0  0.0  0.0          1.0
9524  2021-01-31T20:00:00.0/2021-01-31T21:00:00.0  0.0            0.0            0.0            0.0            0.0  0.0  0.0  0.0  0.0          1.0
9525  2021-01-31T21:00:00.0/2021-01-31T22:00:00.0  0.0            0.0            0.0            0.0            0.0  0.0  0.0  0.0  0.0          1.0
9526  2021-01-31T22:00:00.0/2021-01-31T23:00:00.0  0.0            0.0            0.0            0.0            0.0  0.0  0.0  0.0  0.0          1.0
9527  2021-01-31T23:00:00.0/2021-02-01T00:00:00.0  0.0            0.0            0.0            0.0            0.0  0.0  0.0  0.0  0.0          1.0


In [6]:
periods = df["observation_period"].str.split("/", expand=True)

df["start_time"] = pd.to_datetime(periods[0])
df["end_time"] = pd.to_datetime(periods[1])

df = df.sort_values("start_time").reset_index(drop=True)

print("========== TIMESTAMP PARSING ==========")

print("\nFirst start time:")
print(df["start_time"].iloc[0])

print("\nLast start time:")
print(df["start_time"].iloc[-1])

print("\nFirst end time:")
print(df["end_time"].iloc[0])

print("\nLast end time:")
print(df["end_time"].iloc[-1])

========== TIMESTAMP PARSING ==========

First start time:
2020-01-01 00:00:00

Last start time:
2021-01-31 23:00:00

First end time:
2020-01-01 01:00:00

Last end time:
2021-02-01 00:00:00


In [7]:
duration_hours = (
    (df["end_time"] - df["start_time"])
    .dt.total_seconds() / 3600
)

print("========== OBSERVATION DURATION ==========")
print(duration_hours.value_counts().sort_index())

========== OBSERVATION DURATION ==========
1.0    9528
Name: count, dtype: int64


In [8]:
time_difference = df["start_time"].diff()

unexpected_intervals = (
    time_difference.iloc[1:] != pd.Timedelta(hours=1)
)

print("========== TEMPORAL CONTINUITY ==========")

print(
    "Unexpected intervals:",
    unexpected_intervals.sum()
)

if unexpected_intervals.sum() > 0:
    print("\nUnexpected interval locations:")
    print(
        df.loc[
            unexpected_intervals[unexpected_intervals].index,
            ["start_time", "end_time"]
        ]
    )

========== TEMPORAL CONTINUITY ==========
Unexpected intervals: 0


In [9]:
print("========== DUPLICATE CHECK ==========")

print(
    "Duplicate observation periods:",
    df["observation_period"].duplicated().sum()
)

print(
    "Duplicate start timestamps:",
    df["start_time"].duplicated().sum()
)

========== DUPLICATE CHECK ==========
Duplicate observation periods: 0
Duplicate start timestamps: 0


## Analysis Period Definition

The downloaded dataset extends beyond the primary 2020 calendar year and contains observations through 31 January 2021.

The primary historical analysis period for this project is defined as the complete calendar year:

$$
2020\text{-}01\text{-}01
\leq t <
2021\text{-}01\text{-}01
$$

Observations from January 2021 are retained in the dataset but are not included in the primary 2020 historical analysis.

The additional January 2021 observations may later be used as an out-of-period check of the behavior of thresholds or statistical characteristics derived from the 2020 period.

No observations are deleted from the raw dataset during this separation.

In [10]:
analysis_start = pd.Timestamp("2020-01-01 00:00:00")
analysis_end = pd.Timestamp("2021-01-01 00:00:00")

df["analysis_period"] = np.where(
    (df["start_time"] >= analysis_start) &
    (df["start_time"] < analysis_end),
    "primary_2020",
    "extension_2021"
)

print("========== ANALYSIS PERIOD ==========")
print(df["analysis_period"].value_counts())

========== ANALYSIS PERIOD ==========
analysis_period
primary_2020      8784
extension_2021     744
Name: count, dtype: int64


In [11]:
print("========== PERIOD BOUNDARIES ==========")

for period_name in ["primary_2020", "extension_2021"]:
    period_df = df[df["analysis_period"] == period_name]

    print(f"\n{period_name}")
    print("Rows:", len(period_df))
    print("First:", period_df["start_time"].min())
    print("Last:", period_df["start_time"].max())

========== PERIOD BOUNDARIES ==========

primary_2020
Rows: 8784
First: 2020-01-01 00:00:00
Last: 2020-12-31 23:00:00

extension_2021
Rows: 744
First: 2021-01-01 00:00:00
Last: 2021-01-31 23:00:00


## Historical Radiation Data Quality Assessment

The temporal structure of the historical dataset has been validated.

The next step is to assess the numerical and physical consistency of the radiation measurements across the full downloaded period.

The primary radiation variables are:

- GHI - Global Horizontal Irradiation
- BHI - Beam Irradiation on the Horizontal Plane
- DHI - Diffuse Irradiation on the Horizontal Plane
- BNI - Beam Irradiation on a plane normal to the sun rays

Corresponding clear-sky quantities are also provided by CAMS.

The quality assessment considers:

1. Missing values.
2. Negative radiation values.
3. Reliability values.
4. Consistency between GHI, BHI, and DHI.
5. Observed versus clear-sky radiation.
6. Nighttime behavior.

No observation will be removed during this diagnostic stage.

In [12]:
numeric_columns = [
    "toa",
    "clear_sky_ghi",
    "clear_sky_bhi",
    "clear_sky_dhi",
    "clear_sky_bni",
    "ghi",
    "bhi",
    "dhi",
    "bni",
    "reliability",
]

print("========== MISSING VALUES ==========")

print(
    df[numeric_columns]
    .isna()
    .sum()
)

========== MISSING VALUES ==========
toa              0
clear_sky_ghi    0
clear_sky_bhi    0
clear_sky_dhi    0
clear_sky_bni    0
ghi              0
bhi              0
dhi              0
bni              0
reliability      0
dtype: int64


In [13]:
radiation_columns = [
    "toa",
    "clear_sky_ghi",
    "clear_sky_bhi",
    "clear_sky_dhi",
    "clear_sky_bni",
    "ghi",
    "bhi",
    "dhi",
    "bni",
]

print("========== NEGATIVE VALUES ==========")

for column in radiation_columns:
    count = (df[column] < 0).sum()
    print(f"{column}: {count}")

========== NEGATIVE VALUES ==========
toa: 0
clear_sky_ghi: 0
clear_sky_bhi: 0
clear_sky_dhi: 0
clear_sky_bni: 0
ghi: 0
bhi: 0
dhi: 0
bni: 0


In [14]:
print("========== RELIABILITY ==========")

print(
    df["reliability"].describe()
)

print("\nReliability below 1.0:")
print(
    (df["reliability"] < 1.0).sum()
)

print("\nReliability below 0.9:")
print(
    (df["reliability"] < 0.9).sum()
)

print("\nReliability below 0.8:")
print(
    (df["reliability"] < 0.8).sum()
)

========== RELIABILITY ==========
count    9528.000000
mean        0.976762
std         0.071989
min         0.500000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: reliability, dtype: float64

Reliability below 1.0:
1242

Reliability below 0.9:
848

Reliability below 0.8:
495


In [15]:
df["reliability_flag"] = np.where(
    df["reliability"] < 1.0,
    "reduced_reliability",
    "full_reliability"
)

print("========== RELIABILITY FLAG ==========")

print(
    df["reliability_flag"]
    .value_counts()
)

========== RELIABILITY FLAG ==========
reliability_flag
full_reliability       8286
reduced_reliability    1242
Name: count, dtype: int64


In [16]:
df["ghi_component_difference"] = (
    df["ghi"] -
    (df["bhi"] + df["dhi"])
)

print("========== GHI COMPONENT CONSISTENCY ==========")

print(
    df["ghi_component_difference"]
    .describe()
)

print(
    "\nAbsolute difference > 0.01 Wh/m²:",
    (
        df["ghi_component_difference"].abs() > 0.01
    ).sum()
)

========== GHI COMPONENT CONSISTENCY ==========
count    9.528000e+03
mean     2.413938e-07
std      3.597441e-05
min     -1.000000e-04
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e-04
Name: ghi_component_difference, dtype: float64

Absolute difference > 0.01 Wh/m²: 0


In [17]:
print("========== OBSERVED VS CLEAR-SKY GHI ==========")

print(
    "GHI > clear-sky GHI:",
    (
        df["ghi"] > df["clear_sky_ghi"]
    ).sum()
)

print(
    "GHI = clear-sky GHI:",
    (
        df["ghi"] == df["clear_sky_ghi"]
    ).sum()
)

print(
    "GHI < clear-sky GHI:",
    (
        df["ghi"] < df["clear_sky_ghi"]
    ).sum()
)

========== OBSERVED VS CLEAR-SKY GHI ==========
GHI > clear-sky GHI: 0
GHI = clear-sky GHI: 5845
GHI < clear-sky GHI: 3683


In [18]:
night_mask = df["clear_sky_ghi"] == 0

print("========== NIGHTTIME CONSISTENCY ==========")

print(
    "Nighttime observations:",
    night_mask.sum()
)

print(
    "Nighttime observations with positive GHI:",
    (
        df.loc[night_mask, "ghi"] > 0
    ).sum()
)

print(
    "Nighttime observations with positive BHI:",
    (
        df.loc[night_mask, "bhi"] > 0
    ).sum()
)

print(
    "Nighttime observations with positive DHI:",
    (
        df.loc[night_mask, "dhi"] > 0
    ).sum()
)

print(
    "Nighttime observations with positive BNI:",
    (
        df.loc[night_mask, "bni"] > 0
    ).sum()
)

========== NIGHTTIME CONSISTENCY ==========
Nighttime observations: 4463
Nighttime observations with positive GHI: 0
Nighttime observations with positive BHI: 0
Nighttime observations with positive DHI: 0
Nighttime observations with positive BNI: 0


## Reliability and Solar Conditions

The CAMS dataset provides a reliability value between 0 and 1 for each observation.

A reliability value of 1 indicates the full proportion of data considered reliable in the corresponding time-step summarization, while values below 1 indicate reduced reliability.

The presence of reduced-reliability observations does not by itself establish that the corresponding radiation values are invalid.

Therefore, reliability is treated as a quality indicator rather than an automatic exclusion criterion.

The relationship between reliability and solar conditions will be examined before any filtering policy is defined.

In particular, this analysis will investigate:

- the distribution of reliability across the dataset;
- the relationship between reliability and solar-active periods;
- the relationship between reliability and observed GHI;
- the relationship between reliability and clear-sky GHI;
- whether reduced reliability is concentrated around specific parts of the solar cycle.

No observations will be removed during this investigation.

In [19]:
print("========== RELIABILITY BY ANALYSIS PERIOD ==========")

reliability_by_period = (
    df.groupby("analysis_period")["reliability"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)

print(reliability_by_period)

========== RELIABILITY BY ANALYSIS PERIOD ==========
                 count      mean  median  minimum  q25  q75  maximum
analysis_period                                                     
extension_2021     744  0.973510     1.0      0.5  1.0  1.0      1.0
primary_2020      8784  0.977038     1.0      0.5  1.0  1.0      1.0


In [20]:
print("========== RELIABILITY CATEGORIES ==========")

reliability_bins = pd.cut(
    df["reliability"],
    bins=[-np.inf, 0.5, 0.7, 0.8, 0.9, 0.99, 1.0],
    labels=[
        "<=0.5",
        "0.5-0.7",
        "0.7-0.8",
        "0.8-0.9",
        "0.9-0.99",
        "1.0"
    ],
    include_lowest=True
)

print(
    reliability_bins.value_counts()
    .sort_index()
)

========== RELIABILITY CATEGORIES ==========
reliability
<=0.5          4
0.5-0.7      177
0.7-0.8      338
0.8-0.9      352
0.9-0.99     351
1.0         8306
Name: count, dtype: int64


In [21]:
df["solar_period"] = np.where(
    df["clear_sky_ghi"] == 0,
    "nighttime",
    "daylight"
)

print("========== RELIABILITY BY SOLAR PERIOD ==========")

reliability_solar_period = (
    df.groupby("solar_period")["reliability"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)

print(reliability_solar_period)

========== RELIABILITY BY SOLAR PERIOD ==========
              count      mean  median  minimum  q25  q75  maximum
solar_period                                                     
daylight       5065  0.956298     1.0   0.5000  1.0  1.0      1.0
nighttime      4463  0.999987     1.0   0.9917  1.0  1.0      1.0


In [22]:
print("========== REDUCED RELIABILITY BY SOLAR PERIOD ==========")

print(
    pd.crosstab(
        df["solar_period"],
        df["reliability_flag"]
    )
)

========== REDUCED RELIABILITY BY SOLAR PERIOD ==========
reliability_flag  full_reliability  reduced_reliability
solar_period                                           
daylight                      3830                 1235
nighttime                     4456                    7


In [23]:
print("========== RELIABILITY VS OBSERVED GHI ==========")

print(
    df.groupby("reliability_flag")["ghi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)

========== RELIABILITY VS OBSERVED GHI ==========
                     count        mean    median  minimum       q25       q75  \
reliability_flag                                                                
full_reliability      8286  158.999162   0.00000      0.0  0.000000  271.6900   
reduced_reliability   1242   42.365152  19.76245      0.0  3.926225   53.8796   

                       maximum  
reliability_flag                
full_reliability     1001.8989  
reduced_reliability   754.9455  


In [24]:
print("========== RELIABILITY VS CLEAR-SKY GHI ==========")

print(
    df.groupby("reliability_flag")["clear_sky_ghi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)

========== RELIABILITY VS CLEAR-SKY GHI ==========
                     count        mean    median  minimum       q25  \
reliability_flag                                                      
full_reliability      8286  241.863536   0.00000      0.0  0.000000   
reduced_reliability   1242   66.414888  37.01445      0.0  6.916225   

                            q75    maximum  
reliability_flag                            
full_reliability     450.730550  1002.4860  
reduced_reliability   83.071625   983.3475  


## Reliability Within Daylight Observations

The previous analysis showed that reduced-reliability observations are overwhelmingly concentrated in observations classified as daylight.

However, direct comparisons of GHI between reliability groups can be confounded by the inclusion of nighttime observations, where GHI is zero by definition.

Therefore, the relationship between reliability and solar radiation is examined again using daylight observations only.

This allows the analysis to determine whether reduced reliability is associated with lower observed and clear-sky solar irradiation independently of the nighttime population.

In [25]:
daylight_df = df[
    df["solar_period"] == "daylight"
].copy()

print("========== DAYLIGHT RELIABILITY COMPARISON ==========")

print(
    daylight_df.groupby("reliability_flag")["ghi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)

print("\n========== DAYLIGHT CLEAR-SKY GHI ==========")

print(
    daylight_df.groupby("reliability_flag")["clear_sky_ghi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)

========== DAYLIGHT RELIABILITY COMPARISON ==========
                     count        mean    median  minimum        q25  \
reliability_flag                                                       
full_reliability      3830  343.986177  297.9170   0.0409  150.53960   
reduced_reliability   1235   42.605278   19.9745   0.0024    4.08405   

                            q75    maximum  
reliability_flag                            
full_reliability     491.011425  1001.8989  
reduced_reliability   54.071550   754.9455  

========== DAYLIGHT CLEAR-SKY GHI ==========
                     count        mean    median  minimum        q25  \
reliability_flag                                                       
full_reliability      3830  523.258814  480.5935   0.0409  323.64985   
reduced_reliability   1235   66.791328   37.2161   0.0079    7.07680   

                           q75    maximum  
reliability_flag                           
full_reliability     734.39845  1002.4860  
reduced_re

In [26]:
daylight_df["month"] = daylight_df["start_time"].dt.month

monthly_reliability = (
    daylight_df.groupby("month")
    .agg(
        observations=("reliability", "count"),
        reduced_reliability=(
            "reliability_flag",
            lambda x: (x == "reduced_reliability").sum()
        ),
        mean_reliability=("reliability", "mean")
    )
)

monthly_reliability["reduced_reliability_pct"] = (
    100 *
    monthly_reliability["reduced_reliability"] /
    monthly_reliability["observations"]
)

print("========== DAYLIGHT RELIABILITY BY MONTH ==========")
print(monthly_reliability)

========== DAYLIGHT RELIABILITY BY MONTH ==========
       observations  reduced_reliability  mean_reliability  \
month                                                        
1               593                  165          0.936776   
2               319                   88          0.950733   
3               399                   99          0.959983   
4               435                   98          0.964292   
5               488                   92          0.962570   
6               510                  123          0.966651   
7               509                  109          0.965013   
8               462                   92          0.965928   
9               398                   84          0.964512   
10              365                   93          0.955939   
11              300                  105          0.939254   
12              287                   87          0.932436   

       reduced_reliability_pct  
month                           
1            

In [27]:
daylight_df["hour"] = daylight_df["start_time"].dt.hour

hourly_reliability = (
    daylight_df.groupby("hour")
    .agg(
        observations=("reliability", "count"),
        reduced_reliability=(
            "reliability_flag",
            lambda x: (x == "reduced_reliability").sum()
        ),
        mean_reliability=("reliability", "mean")
    )
)

hourly_reliability["reduced_reliability_pct"] = (
    100 *
    hourly_reliability["reduced_reliability"] /
    hourly_reliability["observations"]
)

print("========== DAYLIGHT RELIABILITY BY HOUR ==========")
print(hourly_reliability)

========== DAYLIGHT RELIABILITY BY HOUR ==========
      observations  reduced_reliability  mean_reliability  \
hour                                                        
3               78                   78          0.849040   
4              161                  161          0.777915   
5              235                  133          0.890221   
6              325                  144          0.916750   
7              397                  146          0.900028   
8              397                   22          0.997147   
9              397                    4          0.998195   
10             397                    5          0.997334   
11             397                   12          0.995592   
12             397                   10          0.997166   
13             397                    6          0.998573   
14             397                    7          0.997796   
15             397                  131          0.938347   
16             295                

## Solar Geometry and the Definition of Solar-Active Periods

The CAMS radiation values are integrated over hourly observation intervals rather than representing instantaneous measurements.

Therefore, classification of solar-active periods should account for the solar geometry over the entire observation interval.

A simple criterion based only on whether the clear-sky GHI is greater than zero can include transition periods with very small solar irradiation.

Similarly, solar elevation evaluated only at the midpoint of an interval may classify an interval incorrectly when the interval spans a sunrise or sunset transition.

Solar position is therefore examined at the beginning and end of each observation interval.

The purpose of this analysis is diagnostic: to understand the relationship between solar geometry, clear-sky irradiation, and the hourly observations before defining the final CSI analysis population.

No final solar-elevation threshold is imposed at this stage.

In [28]:
import pvlib

In [29]:
latitude = 46.80
longitude = 9.83

start_times_utc = df["start_time"].dt.tz_localize("UTC")
end_times_utc = df["end_time"].dt.tz_localize("UTC")

solar_position_start = pvlib.solarposition.get_solarposition(
    time=start_times_utc,
    latitude=latitude,
    longitude=longitude
)

solar_position_end = pvlib.solarposition.get_solarposition(
    time=end_times_utc,
    latitude=latitude,
    longitude=longitude
)

df["solar_elevation_start"] = (
    solar_position_start["elevation"].values
)

df["solar_elevation_end"] = (
    solar_position_end["elevation"].values
)

print("Solar position calculated successfully.")

Solar position calculated successfully.


In [30]:
print("========== SOLAR ELEVATION ==========")

print("\nStart elevation:")
print(
    df["solar_elevation_start"]
    .describe()
)

print("\nEnd elevation:")
print(
    df["solar_elevation_end"]
    .describe()
)

========== SOLAR ELEVATION ==========

Start elevation:
count    9528.000000
mean       -1.095392
std        32.757280
min       -66.344967
25%       -25.475907
50%        -0.896264
75%        22.783528
max        66.208818
Name: solar_elevation_start, dtype: float64

End elevation:
count    9528.000000
mean       -1.094831
std        32.756229
min       -66.344967
25%       -25.475907
50%        -0.896264
75%        22.783528
max        66.208818
Name: solar_elevation_end, dtype: float64


In [31]:
geometry_summary = (
    df.groupby(
        pd.cut(
            df["solar_elevation_start"],
            bins=[
                -90,
                -5,
                0,
                2,
                5,
                10,
                20,
                30,
                90
            ]
        )
    )["clear_sky_ghi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        maximum="max"
    )
)

print("========== CLEAR-SKY GHI BY START ELEVATION ==========")
print(geometry_summary)

========== CLEAR-SKY GHI BY START ELEVATION ==========
                       count        mean    median   minimum    maximum
solar_elevation_start                                                  
(-90, -5]               4401    0.113652    0.0000    0.0000    13.1936
(-5, 0]                  444   15.555530    0.0000    0.0000    67.0229
(0, 2]                   194   32.370264    1.1978    0.0000    96.7702
(2, 5]                   241   61.544364   86.3953    1.4200   147.6575
(5, 10]                  427  114.466285  140.5863   10.4468   250.5017
(10, 20]                1069  251.098689  265.3440   54.6998   453.0837
(20, 30]                 947  419.803247  424.5998  207.4103   630.7218
(30, 90]                1805  744.152027  747.1271  382.2886  1002.4860


C:\Users\Admin\AppData\Local\Temp\ipykernel_10424\1833812579.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(


## Solar-Geometry Coverage of Hourly Intervals

The CAMS radiation values represent irradiation integrated over hourly observation intervals.

Consequently, the solar elevation at a single timestamp cannot fully describe whether an observation interval contains solar radiation.

An hourly interval may begin with the sun below the horizon and end with the sun above the horizon, or the reverse may occur around sunrise and sunset.

To characterize these transition intervals, solar elevation is evaluated at multiple points within each hourly observation interval.

The resulting solar-geometry information is used to distinguish:

- intervals that remain below the horizon;
- intervals that contain a sunrise or sunset transition;
- intervals that are predominantly solar-active.

This analysis is diagnostic and does not yet impose a final solar-elevation threshold for drought analysis.

In [32]:
interval_times = pd.DataFrame({
    "t0": df["start_time"],
    "t15": df["start_time"] + pd.Timedelta(minutes=15),
    "t30": df["start_time"] + pd.Timedelta(minutes=30),
    "t45": df["start_time"] + pd.Timedelta(minutes=45),
    "t60": df["end_time"]
})

for column in interval_times.columns:
    interval_times[column] = (
        interval_times[column]
        .dt.tz_localize("UTC")
    )

elevation_points = {}

for column in interval_times.columns:
    solar_position = pvlib.solarposition.get_solarposition(
        time=interval_times[column],
        latitude=latitude,
        longitude=longitude
    )
    
    elevation_points[column] = solar_position["elevation"].values

df["elevation_t0"] = elevation_points["t0"]
df["elevation_t15"] = elevation_points["t15"]
df["elevation_t30"] = elevation_points["t30"]
df["elevation_t45"] = elevation_points["t45"]
df["elevation_t60"] = elevation_points["t60"]

print("Solar elevation evaluated at five points per hourly interval.")

Solar elevation evaluated at five points per hourly interval.


In [33]:
elevation_columns = [
    "elevation_t0",
    "elevation_t15",
    "elevation_t30",
    "elevation_t45",
    "elevation_t60"
]

df["solar_elevation_min"] = df[elevation_columns].min(axis=1)
df["solar_elevation_max"] = df[elevation_columns].max(axis=1)

print("========== INTERVAL SOLAR GEOMETRY ==========")

print(
    df[
        [
            "start_time",
            "clear_sky_ghi",
            "solar_elevation_min",
            "solar_elevation_max"
        ]
    ].head(10)
)

========== INTERVAL SOLAR GEOMETRY ==========
           start_time  clear_sky_ghi  solar_elevation_min  solar_elevation_max
0 2020-01-01 00:00:00         0.0000           -65.165557           -59.391497
1 2020-01-01 01:00:00         0.0000           -59.391497           -50.772015
2 2020-01-01 02:00:00         0.0000           -50.772015           -40.941394
3 2020-01-01 03:00:00         0.0000           -40.941394           -30.713314
4 2020-01-01 04:00:00         0.0000           -30.713314           -20.538903
5 2020-01-01 05:00:00         0.0000           -20.538903           -10.747120
6 2020-01-01 06:00:00         0.0000           -10.747120            -1.651971
7 2020-01-01 07:00:00        36.0358            -1.651971             6.393773
8 2020-01-01 08:00:00       169.2675             6.393773            12.976214
9 2020-01-01 09:00:00       287.6712            12.976214            17.639104


In [34]:
df["geometry_class"] = np.select(
    [
        df["solar_elevation_max"] <= 0,
        df["solar_elevation_min"] < 0,
        df["solar_elevation_min"] >= 0
    ],
    [
        "night",
        "transition",
        "solar_active"
    ],
    default="check"
)

print("========== GEOMETRY CLASS ==========")
print(
    df["geometry_class"]
    .value_counts()
)

========== GEOMETRY CLASS ==========
geometry_class
night           4448
solar_active    4286
transition       794
Name: count, dtype: int64


In [35]:
geometry_ghi_summary = (
    df.groupby("geometry_class", observed=False)["clear_sky_ghi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)

print("========== CLEAR-SKY GHI BY GEOMETRY CLASS ==========")
print(geometry_ghi_summary)

========== CLEAR-SKY GHI BY GEOMETRY CLASS ==========
                count        mean     median  minimum         q25         q75  \
geometry_class                                                                  
night            4448    0.000000    0.00000   0.0000    0.000000    0.000000   
solar_active     4286  483.531738  443.52580  44.0608  268.753325  707.515300   
transition        794   17.823073   10.07335   0.0000    1.734025   33.448725   

                  maximum  
geometry_class             
night              0.0000  
solar_active    1002.4860  
transition        67.3095  


In [36]:
transition_df = df[
    df["geometry_class"] == "transition"
].copy()

print("========== TRANSITION INTERVALS ==========")

print(
    transition_df[
        [
            "start_time",
            "clear_sky_ghi",
            "ghi",
            "solar_elevation_min",
            "solar_elevation_max",
            "reliability"
        ]
    ].head(30)
)

========== TRANSITION INTERVALS ==========
             start_time  clear_sky_ghi      ghi  solar_elevation_min  \
7   2020-01-01 07:00:00        36.0358  36.0358            -1.651971   
15  2020-01-01 15:00:00        18.9517  18.9517            -3.378235   
31  2020-01-02 07:00:00        35.2385  35.2385            -1.655565   
39  2020-01-02 15:00:00        18.2993  18.2993            -3.243240   
55  2020-01-03 07:00:00        31.0171  26.0812            -1.652506   
63  2020-01-03 15:00:00        18.0016  18.0016            -3.103255   
79  2020-01-04 07:00:00        33.7980  23.2215            -1.642760   
87  2020-01-04 15:00:00        20.4160  13.2926            -2.958381   
103 2020-01-05 07:00:00        35.7713  35.7713            -1.626302   
111 2020-01-05 15:00:00        23.7961  23.7961            -2.808716   
127 2020-01-06 07:00:00        35.9691  35.9691            -1.603110   
135 2020-01-06 15:00:00        25.2343  25.2343            -2.654360   
151 2020-01-07 07:00:

In [37]:
print("\nTransition count:", len(transition_df))

print(
    "\nTransition clear-sky GHI:"
)

print(
    transition_df["clear_sky_ghi"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)


Transition count: 794

Transition clear-sky GHI:
count    794.000000
mean      17.823073
std       17.797466
min        0.000000
10%        0.257540
25%        1.734025
50%       10.073350
75%       33.448725
90%       45.567870
95%       50.870255
max       67.309500
Name: clear_sky_ghi, dtype: float64


## Transition Intervals and Clear-Sky Irradiation

The transition class contains hourly intervals in which the solar elevation changes from below to above the horizon, or from above to below the horizon.

Because the radiation variable is an integrated hourly quantity, the transition class may contain both zero and positive clear-sky irradiation.

The distribution of clear-sky GHI within this class is therefore examined before determining how transition intervals will be treated in the primary historical analysis.

Transition observations are retained in the dataset regardless of their classification.

In [38]:
transition_ghi = transition_df["clear_sky_ghi"]

print("========== TRANSITION IRRADIATION ==========")

print(
    "Transition observations:",
    len(transition_df)
)

print(
    "Clear-sky GHI = 0:",
    (transition_ghi == 0).sum()
)

print(
    "Clear-sky GHI > 0:",
    (transition_ghi > 0).sum()
)

print(
    "Clear-sky GHI > 20:",
    (transition_ghi > 20).sum()
)

print(
    "Clear-sky GHI > 50:",
    (transition_ghi > 50).sum()
)

print("\nPercentage distribution:")

print(
    "Zero:",
    100 * (transition_ghi == 0).mean()
)

print(
    "> 0:",
    100 * (transition_ghi > 0).mean()
)

print(
    "> 20:",
    100 * (transition_ghi > 20).mean()
)

print(
    "> 50:",
    100 * (transition_ghi > 50).mean()
)

========== TRANSITION IRRADIATION ==========
Transition observations: 794
Clear-sky GHI = 0: 15
Clear-sky GHI > 0: 779
Clear-sky GHI > 20: 298
Clear-sky GHI > 50: 46

Percentage distribution:
Zero: 1.8891687657430731
> 0: 98.11083123425692
> 20: 37.531486146095716
> 50: 5.793450881612091


In [39]:
high_transition = (
    transition_df
    .sort_values("clear_sky_ghi", ascending=False)
    [
        [
            "start_time",
            "clear_sky_ghi",
            "ghi",
            "solar_elevation_min",
            "solar_elevation_max",
            "reliability"
        ]
    ]
    .head(20)
)

print("========== HIGHEST-IRRADIATION TRANSITION INTERVALS ==========")
print(high_transition.to_string(index=False))

========== HIGHEST-IRRADIATION TRANSITION INTERVALS ==========
         start_time  clear_sky_ghi     ghi  solar_elevation_min  solar_elevation_max  reliability
2020-02-28 16:00:00        67.3095 67.3095            -0.183294             9.606774       0.7917
2020-03-03 06:00:00        67.0229  9.9626            -0.172460             9.740641       0.7333
2020-04-03 05:00:00        66.9257 17.5264            -0.077697            10.177696       0.7250
2020-09-29 16:00:00        61.6011 47.8130            -0.189736             9.939828       0.9167
2020-01-20 15:00:00        61.4624 61.4624            -0.054955             8.256223       0.6583
2020-04-02 05:00:00        60.4048 60.4048            -0.408039             9.849083       0.7417
2020-01-23 07:00:00        60.2436 60.2436            -0.181156             8.258464       0.7333
2021-01-23 07:00:00        60.0863 59.1121            -0.074357             8.380104       0.7250
2020-02-27 16:00:00        59.5933  9.6470            -

In [40]:
zero_transition = (
    transition_df[
        transition_df["clear_sky_ghi"] == 0
    ]
    [
        [
            "start_time",
            "clear_sky_ghi",
            "ghi",
            "solar_elevation_min",
            "solar_elevation_max",
            "reliability"
        ]
    ]
    .head(20)
)

print("========== ZERO-IRRADIATION TRANSITIONS ==========")
print(zero_transition.to_string(index=False))

========== ZERO-IRRADIATION TRANSITIONS ==========
         start_time  clear_sky_ghi  ghi  solar_elevation_min  solar_elevation_max  reliability
2020-01-25 06:00:00            0.0  0.0            -9.312372             0.111412       1.0000
2020-02-29 17:00:00            0.0  0.0           -10.126534             0.064105       0.9917
2020-03-04 05:00:00            0.0  0.0           -10.096123             0.146596       1.0000
2020-04-12 18:00:00            0.0  0.0            -9.643676             0.052324       0.9917
2020-05-31 19:00:00            0.0  0.0            -7.883473             0.105648       0.9917
2020-07-20 19:00:00            0.0  0.0            -8.175000             0.105723       0.9917
2020-07-26 03:00:00            0.0  0.0            -8.366787             0.079885       1.0000
2020-08-29 18:00:00            0.0  0.0            -9.687883             0.034693       0.9917
2020-09-12 04:00:00            0.0  0.0           -10.008444             0.033875       1.0000

## Operational Definition of the CSI Analysis Population

The solar-geometry analysis demonstrates that hourly intervals near sunrise and sunset can contain substantial integrated solar irradiation even when the solar elevation crosses the horizon within the interval.

Therefore, transition intervals are not automatically excluded from the CSI analysis.

For the historical solar-availability series, the mathematical CSI population is defined by the CAMS clear-sky reference:

$$
GHI_{\text{clear},t} > 0
$$

For these observations:

$$
CSI_t =
\frac{GHI_{\text{observed},t}}
{GHI_{\text{clear},t}}
$$

Observations with:

$$
GHI_{\text{clear},t} = 0
$$

are assigned no CSI value because the normalization denominator is zero.

Solar geometry is retained as an additional diagnostic classification:

- `night`: solar elevation remains at or below the horizon across the sampled interval;
- `transition`: the interval crosses the horizon;
- `solar_active`: solar elevation remains above the horizon across the sampled interval.

Transition observations are retained because many contain non-negligible integrated solar irradiation.

The geometry classification is therefore used to support sensitivity analysis rather than to automatically exclude observations.

No arbitrary clear-sky GHI threshold is imposed to define the CSI population.

In [41]:
df["csi"] = np.nan

valid_reference = df["clear_sky_ghi"] > 0

df.loc[valid_reference, "csi"] = (
    df.loc[valid_reference, "ghi"]
    /
    df.loc[valid_reference, "clear_sky_ghi"]
)

print("========== HISTORICAL CSI ==========")

print(
    "Total observations:",
    len(df)
)

print(
    "Positive clear-sky reference:",
    valid_reference.sum()
)

print(
    "Zero clear-sky reference:",
    (~valid_reference).sum()
)

print(
    "Valid CSI observations:",
    df["csi"].notna().sum()
)

print(
    "CSI missing:",
    df["csi"].isna().sum()
)

========== HISTORICAL CSI ==========
Total observations: 9528
Positive clear-sky reference: 5065
Zero clear-sky reference: 4463
Valid CSI observations: 5065
CSI missing: 4463


In [42]:
valid_csi = df[df["csi"].notna()].copy()

print("========== HISTORICAL CSI VALIDATION ==========")

print("\nCSI distribution:")
print(
    valid_csi["csi"].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print(
    "\nCSI outside [0, 1]:",
    (
        (valid_csi["csi"] < 0) |
        (valid_csi["csi"] > 1)
    ).sum()
)

print(
    "\nCSI = 1:",
    (valid_csi["csi"] == 1).sum()
)

print(
    "CSI < 0.2:",
    (valid_csi["csi"] < 0.2).sum()
)

print(
    "CSI < 0.5:",
    (valid_csi["csi"] < 0.5).sum()
)

========== HISTORICAL CSI VALIDATION ==========

CSI distribution:
count    5065.000000
mean        0.683697
std         0.319477
min         0.062676
1%          0.083722
5%          0.129295
10%         0.170662
25%         0.384579
50%         0.777589
75%         1.000000
90%         1.000000
95%         1.000000
99%         1.000000
max         1.000000
Name: csi, dtype: float64

CSI outside [0, 1]: 0

CSI = 1: 1382
CSI < 0.2: 619
CSI < 0.5: 1571


In [43]:
csi_geometry_summary = (
    valid_csi
    .groupby("geometry_class", observed=False)["csi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        maximum="max"
    )
)

print("========== CSI BY GEOMETRY CLASS ==========")
print(csi_geometry_summary)

========== CSI BY GEOMETRY CLASS ==========
                count      mean    median   minimum       q25      q75  \
geometry_class                                                           
solar_active     4286  0.667186  0.743058  0.062676  0.373787  0.99059   
transition        779  0.774534  1.000000  0.111365  0.592660  1.00000   

                maximum  
geometry_class           
solar_active        1.0  
transition          1.0  


In [44]:
print("========== CSI POPULATION BY GEOMETRY ==========")

print(
    valid_csi["geometry_class"]
    .value_counts()
)

print("\nPercentages:")
print(
    100 *
    valid_csi["geometry_class"]
    .value_counts(normalize=True)
)

========== CSI POPULATION BY GEOMETRY ==========
geometry_class
solar_active    4286
transition       779
Name: count, dtype: int64

Percentages:
geometry_class
solar_active    84.619941
transition      15.380059
Name: proportion, dtype: float64


## Historical CSI Seasonality

Before defining anomaly thresholds, the temporal structure of the normalized CSI series must be examined.

Clear-sky normalization reduces the deterministic influence of solar geometry, but the distribution of observed solar availability may still vary across seasons and times of day.

A single global threshold may therefore produce different levels of rarity across the year.

The CSI distribution is consequently examined by:

- month;
- hour of day;
- solar-geometry class.

This analysis is descriptive and does not define the final anomaly or drought threshold.

In [45]:
valid_csi["month"] = valid_csi["start_time"].dt.month

monthly_csi = (
    valid_csi
    .groupby("month")["csi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        q01=lambda x: x.quantile(0.01),
        q05=lambda x: x.quantile(0.05),
        q10=lambda x: x.quantile(0.10),
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        minimum="min",
        maximum="max"
    )
)

print("========== MONTHLY CSI DISTRIBUTION ==========")
print(monthly_csi)

========== MONTHLY CSI DISTRIBUTION ==========
       count      mean    median       q01       q05       q10       q25  \
month                                                                      
1        593  0.702065  0.923722  0.102712  0.143409  0.164207  0.336320   
2        319  0.658735  0.754642  0.100587  0.140425  0.166203  0.264287   
3        399  0.721402  0.891528  0.116321  0.155103  0.235646  0.429718   
4        435  0.762220  0.802418  0.118724  0.194833  0.377937  0.681251   
5        488  0.661007  0.735709  0.095899  0.135828  0.179271  0.411046   
6        510  0.578232  0.566179  0.066152  0.116124  0.146312  0.259532   
7        509  0.724620  0.831600  0.091538  0.142923  0.227656  0.499073   
8        462  0.680927  0.820711  0.073005  0.111729  0.152019  0.354231   
9        398  0.725361  0.840693  0.072588  0.106533  0.233039  0.492610   
10       365  0.565036  0.590654  0.076561  0.118916  0.129641  0.304805   
11       300  0.846756  0.968529  0.12482

In [46]:
monthly_low_csi = (
    valid_csi
    .groupby("month")
    .agg(
        observations=("csi", "count"),
        csi_below_02=(
            "csi",
            lambda x: (x < 0.2).sum()
        ),
        csi_below_05=(
            "csi",
            lambda x: (x < 0.5).sum()
        ),
        csi_equal_1=(
            "csi",
            lambda x: (x == 1.0).sum()
        )
    )
)

monthly_low_csi["below_02_pct"] = (
    100 *
    monthly_low_csi["csi_below_02"] /
    monthly_low_csi["observations"]
)

monthly_low_csi["below_05_pct"] = (
    100 *
    monthly_low_csi["csi_below_05"] /
    monthly_low_csi["observations"]
)

monthly_low_csi["equal_1_pct"] = (
    100 *
    monthly_low_csi["csi_equal_1"] /
    monthly_low_csi["observations"]
)

print("========== MONTHLY CSI LOW-TAIL FREQUENCY ==========")
print(monthly_low_csi)

========== MONTHLY CSI LOW-TAIL FREQUENCY ==========
       observations  csi_below_02  csi_below_05  csi_equal_1  below_02_pct  \
month                                                                        
1               593            83           189          276     13.996627   
2               319            57           117          116     17.868339   
3               399            34           110          162      8.521303   
4               435            23            67           92      5.287356   
5               488            57           152           63     11.680328   
6               510            80           230           88     15.686275   
7               509            44           130          129      8.644401   
8               462            64           155          136     13.852814   
9               398            37           103          129      9.296482   
10              365            64           149           24     17.534247   
11         

In [47]:
valid_csi["hour"] = valid_csi["start_time"].dt.hour

hourly_csi = (
    valid_csi
    .groupby("hour")["csi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        q01=lambda x: x.quantile(0.01),
        q05=lambda x: x.quantile(0.05),
        q10=lambda x: x.quantile(0.10),
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        minimum="min",
        maximum="max"
    )
)

print("========== HOURLY CSI DISTRIBUTION ==========")
print(hourly_csi)

========== HOURLY CSI DISTRIBUTION ==========
      count      mean    median       q01       q05       q10       q25  \
hour                                                                      
3        78  0.694808  1.000000  0.123600  0.129456  0.134080  0.365701   
4       161  0.726230  0.996029  0.116481  0.126720  0.135431  0.428755   
5       235  0.665966  0.894080  0.098418  0.118481  0.123846  0.217892   
6       325  0.677901  0.850233  0.086763  0.112481  0.129940  0.291570   
7       397  0.695202  0.823999  0.081261  0.123163  0.151556  0.368486   
8       397  0.677632  0.790204  0.072567  0.115461  0.138282  0.364712   
9       397  0.654254  0.727545  0.066935  0.104471  0.147878  0.345229   
10      397  0.642045  0.706417  0.068935  0.127657  0.173916  0.320856   
11      397  0.644529  0.711705  0.093734  0.146336  0.201001  0.335259   
12      397  0.653794  0.703340  0.099422  0.161244  0.223394  0.353648   
13      397  0.647779  0.694605  0.101411  0.168092  0

In [48]:
geometry_reliability_csi = (
    valid_csi
    .groupby(
        ["geometry_class", "reliability_flag"],
        observed=False
    )["csi"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        q05=lambda x: x.quantile(0.05),
        q10=lambda x: x.quantile(0.10),
        q25=lambda x: x.quantile(0.25),
        minimum="min",
        maximum="max"
    )
)

print("========== CSI BY GEOMETRY AND RELIABILITY ==========")
print(geometry_reliability_csi)

========== CSI BY GEOMETRY AND RELIABILITY ==========
                                    count      mean    median       q05  \
geometry_class reliability_flag                                           
solar_active   full_reliability      3818  0.667180  0.740179  0.130000   
               reduced_reliability    468  0.667239  0.772444  0.121469   
transition     full_reliability        12  0.965541  1.000000  0.795834   
               reduced_reliability    767  0.771546  1.000000  0.132986   

                                         q10       q25   minimum  maximum  
geometry_class reliability_flag                                            
solar_active   full_reliability     0.190231  0.375591  0.062676      1.0  
               reduced_reliability  0.136623  0.362060  0.079399      1.0  
transition     full_reliability     0.837202  1.000000  0.767382      1.0  
               reduced_reliability  0.149544  0.582612  0.111365      1.0  


# Conclusion

This notebook prepared and evaluated the historical CAMS solar-radiation dataset for subsequent statistical analysis of solar-availability anomalies and drought events.

The downloaded dataset contains 9,528 hourly observations spanning 1 January 2020 through 31 January 2021. The primary historical analysis period was defined as the complete 2020 calendar year, containing 8,784 hourly observations. The additional January 2021 observations were retained as an extension period and were not used to define the primary 2020 historical distribution.

### Temporal and numerical quality

The historical dataset contains continuous one-hour observation intervals without duplicate observation periods or unexpected temporal gaps.

No missing values were detected in the primary radiation variables or reliability variable. No negative values were identified in the radiation variables.

The relationship between global, beam, and diffuse horizontal irradiation was internally consistent within the numerical precision of the dataset.

### Reliability

CAMS reliability information was retained as a quality-control variable rather than being used as an automatic exclusion criterion.

Reduced reliability occurred predominantly during daylight observations and was strongly concentrated around the lower-solar portions of the daily cycle.

Within solar-active observations, the mean CSI was approximately identical for the full- and reduced-reliability groups in this dataset. This descriptive result does not establish that reliability has no influence on individual observations, but it provides no basis for automatically removing reduced-reliability observations from the analysis.

### Solar geometry

Solar position was evaluated at five points within each hourly observation interval.

Three diagnostic geometry classes were defined:

- `night`: solar elevation remained at or below the horizon;
- `transition`: the interval crossed the horizon;
- `solar_active`: solar elevation remained above the horizon.

Transition intervals were found to contain substantial integrated solar irradiation. Consequently, transition intervals were retained rather than being automatically excluded from the CSI series.

### Clear-sky normalization

The historical Clear-Sky Index was calculated as:

$$
CSI_t =
\frac{GHI_{\text{observed},t}}
{GHI_{\text{clear},t}}
$$

for observations with a positive clear-sky reference.

Of the 9,528 historical observations, 5,065 had a positive clear-sky reference and therefore received a CSI value. The remaining 4,463 observations had zero clear-sky GHI and were assigned no CSI value.

The resulting CSI values ranged from approximately 0.0627 to 1.0000, with no values outside the expected interval:

$$
0 \leq CSI_t \leq 1
$$

The CSI calculation was therefore numerically consistent with the corresponding observed and clear-sky GHI values.

### Temporal structure of CSI

The historical CSI distribution was found to vary substantially across months and hours of the day.

For example, the monthly median CSI ranged from approximately 0.555 to 0.969, while the proportion of observations with CSI below 0.2 varied considerably between months.

Hourly CSI distributions also differed, although some early- and late-day hours contained relatively few observations because of the seasonal solar cycle.

These results demonstrate that the normalized CSI series cannot automatically be assumed to have a stationary distribution across the entire year.

### Implications for anomaly detection

The analyses in this notebook demonstrate that a simple global CSI threshold may not provide an equivalent measure of anomaly severity throughout the year.

The historical dataset therefore requires further investigation of conditional reference distributions before defining a final anomaly threshold.

Candidate approaches include:

1. a global empirical percentile threshold;
2. a seasonally conditioned percentile threshold;
3. a season-and-hour conditioned threshold;
4. a solar-geometry-conditioned reference;
5. a smoothed seasonal and diurnal reference.

These approaches should be compared rather than selecting a threshold arbitrarily.

The current dataset contains only one complete calendar year in the primary analysis period. Therefore, it is not considered sufficient by itself to establish a robust multi-year climatological rarity threshold for solar droughts.

### Transition to anomaly analysis

The next stage of the project will investigate statistical methods for identifying anomalously low solar availability.

The analysis will first examine the suitability of candidate reference distributions and anomaly definitions.

No solar-drought threshold is established in this notebook.

The methodological sequence is therefore:

$$
\text{Historical data}
\rightarrow
\text{Quality control}
\rightarrow
\text{Solar geometry}
\rightarrow
\text{CSI normalization}
\rightarrow
\text{Distributional analysis}
\rightarrow
\text{Anomaly definition}
\rightarrow
\text{Persistence}
\rightarrow
\text{Drought events}
$$

The prepared historical CSI series is therefore suitable for proceeding to the anomaly-detection methodology, while recognizing that final climatological drought claims require a longer historical record.